# Notebook 0 — Download JWST data from MAST

This notebook is a simple first step for a JWST reduction project.

Goal:
1. Search for JWST observations of a target.
2. Select useful data products.
3. Download the files.
4. Open one FITS file to check that everything worked.

This notebook is intentionally simple. More detailed filtering and pipeline reduction will come later.

## 0. Install packages

Run this cell once if the packages are not already installed.

In [1]:
# Uncomment if needed
# !pip install astroquery astropy tqdm

## 1. Imports

In [6]:
from astroquery.mast import Observations
from astropy.io import fits
from pathlib import Path
import glob

## 2. Choose a target

Edit the target name below.


In [11]:
target_name = "TOI-1130b"   # EDIT THIS

download_dir = Path("/media/peng/KINGSTON")
download_dir.mkdir(exist_ok=True)

print(f"Target: {target_name}")
print(f"Download folder: {download_dir.resolve()}")

Target: TOI-1130b
Download folder: /media/peng/KINGSTON


## 3. Search observations on MAST

This searches all observations around the target name, then keeps only JWST observations.

In [12]:
obs_table = Observations.query_object(target_name)

jwst_obs = obs_table[obs_table["obs_collection"] == "JWST"]

print(f"Total observations found: {len(obs_table)}")
print(f"JWST observations found: {len(jwst_obs)}")

jwst_obs[:10]

ConnectionError: HTTPConnectionPool(host='mastresolver.stsci.edu', port=80): Max retries exceeded with url: /Santa-war/query?outputFormat=json&resolveAll=False&name=TOI-1130b (Caused by NameResolutionError("HTTPConnection(host='mastresolver.stsci.edu', port=80): Failed to resolve 'mastresolver.stsci.edu' ([Errno -2] Name or service not known)"))

## 4. Optional: inspect the observations

Useful columns include:
- `obs_id`
- `instrument_name`
- `target_name`
- `filters`
- `t_exptime`

In [ ]:
cols = ["obs_id", "target_name", "instrument_name", "filters", "t_exptime"]

available_cols = [c for c in cols if c in jwst_obs.colnames]
jwst_obs[available_cols][:20]

## 5. Get the list of available data products

Each JWST observation has many associated files.

In [5]:
products = Observations.get_product_list(jwst_obs)

print(f"Number of products: {len(products)}")
products[:10]

NameError: name 'jwst_obs' is not defined

## 6. Select useful FITS files

For a first reduction project, start with:

- `UNCAL`: raw detector data, used as input to JWST Stage 1
- `RATEINTS`: partially reduced integration-level data, useful for faster tests

Later notebooks can also use `CALINTS` and `X1DINTS`.

In [4]:
selected = Observations.filter_products(
    products,
    productSubGroupDescription=["RATEINTS"],
    extension="fits"
)

print(f"Selected FITS files: {len(selected)}")
selected[:20]

NameError: name 'products' is not defined

## 7. Optional: select only one instrument

Use this only if the target has both NIRISS and NIRSpec observations.

Leave this cell unchanged if you want to download everything selected above.

In [10]:
# Need to rerun previous cell after making changes to this

# Example: keep only NIRSpec files: change NIRSPEC --> nrs1 or nrs2, removed .upper()
# selected = selected[["nrs" in str(row["obs_id"]) for row in selected]]

# Example: keep only NIRISS files: change NIRISS --> nis, removed .upper()
selected = selected[["nis" in str(row["obs_id"]) for row in selected]]

print(f"Files still selected: {len(selected)}")

Files still selected: 23


## 8. Download the files

This may take time depending on the number of files.

`cache=True` avoids downloading the same files again if they already exist.

In [ ]:
manifest = Observations.download_products(
    selected,
    download_dir=str(download_dir),
    cache=True
)

print("Download complete.")
manifest[:10]

## 9. List downloaded FITS files

In [10]:
fits_files = sorted(glob.glob(str(download_dir / "**" / "*.fits"), recursive=True))

print(f"Number of downloaded FITS files: {len(fits_files)}")
fits_files[:]

Number of downloaded FITS files: 16


['/media/peng/KINGSTON/mastDownload/JWST/jw03385001001_02101_00001-seg001_nrs1/jw03385001001_02101_00001-seg001_nrs1_uncal.fits',
 '/media/peng/KINGSTON/mastDownload/JWST/jw03385001001_02101_00002-seg001_nrs1/jw03385001001_02101_00002-seg001_nrs1_uncal.fits',
 '/media/peng/KINGSTON/mastDownload/JWST/jw03385001001_04102_00001-seg001_nrs1/jw03385001001_04102_00001-seg001_nrs1_uncal.fits',
 '/media/peng/KINGSTON/mastDownload/JWST/jw03385001001_04102_00001-seg001_nrs2/jw03385001001_04102_00001-seg001_nrs2_uncal.fits',
 '/media/peng/KINGSTON/mastDownload/JWST/jw03385001001_04103_00001-seg001_nrs1/jw03385001001_04103_00001-seg001_nrs1_uncal.fits',
 '/media/peng/KINGSTON/mastDownload/JWST/jw03385001001_04103_00001-seg001_nrs2/jw03385001001_04103_00001-seg001_nrs2_uncal.fits',
 '/media/peng/KINGSTON/mastDownload/JWST/jw03385001001_04104_00001-seg001_nrs1/jw03385001001_04104_00001-seg001_nrs1_uncal.fits',
 '/media/peng/KINGSTON/mastDownload/JWST/jw03385001001_04104_00001-seg001_nrs2/jw033850010

## 10. Open one FITS file

This is just a quick check that the file is readable.

In [11]:
if len(fits_files) == 0:
    print("No FITS files found. Check the download step above.")
else:
    test_file = fits_files[0]
    print(f"Opening: {test_file}")

    with fits.open(test_file) as hdul:
        hdul.info()

Opening: /media/peng/KINGSTON/mastDownload/JWST/jw03385001001_02101_00001-seg001_nrs1/jw03385001001_02101_00001-seg001_nrs1_uncal.fits
Filename: /media/peng/KINGSTON/mastDownload/JWST/jw03385001001_02101_00001-seg001_nrs1/jw03385001001_02101_00001-seg001_nrs1_uncal.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU     196   ()      
  1  SCI           1 ImageHDU        80   (32, 32, 3, 1)   int16 (rescales to uint16)   
  2  GROUP         1 BinTableHDU     38   1R x 13C   [J, I, I, J, I, 26A, I, I, I, I, 36A, D, D]   
  3  INT_TIMES     1 BinTableHDU     24   1R x 7C   [J, D, D, D, D, D, D]   
  4  TARG_ACQ      1 BinTableHDU     67   0R x 0C   []   
  5  ASDF          1 BinTableHDU     11   1R x 1C   [12405B]   


## 11. Check the FITS header

The primary header contains useful information such as the instrument, detector, filter, and program ID.

In [12]:
if len(fits_files) > 0:
    with fits.open(fits_files[0]) as hdul:
        header = hdul[0].header

    keys = [
        "TELESCOP", "INSTRUME", "DETECTOR", "FILTER", "GRATING",
        "PROGRAM", "OBSERVTN", "VISIT", "TARGNAME", "EXP_TYPE",
        "INTSTART", "INTEND"
    ]

    for key in keys:
        if key in header:
            print(f"{key:10s} = {header[key]}")

TELESCOP   = JWST
INSTRUME   = NIRSPEC
DETECTOR   = NRS1
FILTER     = F110W
GRATING    = MIRROR
PROGRAM    = 03385
OBSERVTN   = 001
VISIT      = 001
TARGNAME   = 2MASS J19053016-4126418
EXP_TYPE   = NRS_WATA
INTSTART   = 1
INTEND     = 1


## Summary

At this point, you should have:

1. Found JWST observations for the target.
2. Selected `UNCAL` and/or `RATEINTS` FITS files.
3. Downloaded the files locally.
4. Opened one file with `astropy.io.fits`.

